In [ ]:
!pip install --upgrade pip

In [ ]:
!pip install numpy==2.1.3

In [ ]:
!pip install pandas==2.2.3

In [ ]:
!pip install matplotlib==3.9.2

In [ ]:
!pip install tabulate==0.9.0

In [ ]:
!pip freeze

In [ ]:
import re
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate

In [ ]:
project_id: int = 0 # @TODO: Set a project ID.

In [ ]:
conn = sqlite3.connect('../database/db.sqlite')

## Number of test in-/exclusions

In [ ]:
query = f"SELECT project_id, count(DISTINCT test_class_qualified_name) AS test_class_count, count(*) AS test_method_count, sum(is_included) AS included_count FROM test WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['test_method_count'] - df['included_count']
df

## Number of assertion in-/exclusions

In [ ]:
query = f"SELECT project_id, count(DISTINCT test_id) test_count, count(*) AS assertion_count, sum(is_included) AS included_count FROM assertion WHERE project_id = {project_id} GROUP BY project_id"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['assertion_count'] - df['included_count']
df

The number of tests is usually lower here than above due to tests that do not contain any assertions. 

## Number of generalization in-/exclusions

In [ ]:
query = f"SELECT project_id, variant, count(*) AS total_count, sum(is_included) AS included_count FROM generalization WHERE project_id = {project_id} GROUP BY project_id, variant"
df = pd.read_sql_query(query, conn)
df['excluded_count'] = df['total_count'] - df['included_count']
df

## Number of test / assertion / generalization exclusions per task

In [ ]:
query = f"""
SELECT project_id, 1 AS type_id, 'TEST' AS item_type, '-' AS variant, exclusion_info FROM test WHERE project_id = {project_id} AND is_included = 0
UNION ALL
SELECT project_id, 2, 'ASSERTION', '-', exclusion_info FROM assertion WHERE project_id = {project_id} AND is_included = 0
UNION ALL
SELECT project_id, 3, 'GENERALIZATION', variant, exclusion_info FROM generalization WHERE project_id = {project_id} AND is_included = 0
"""
df = pd.read_sql_query(query, conn)

def extract_task_name(s):
    match = re.search(r'\b(\w+){', s)
    return match.group(1) if match else None

df['exclusion_info'] = df['exclusion_info'].apply(extract_task_name)

df = df.pivot_table(index=['project_id', 'type_id', 'item_type', 'variant'], columns='exclusion_info', aggfunc='size', fill_value=0)
df['Total Exclusions'] = df.sum(axis=1)
df = df[['Total Exclusions'] + [col for col in df if col != 'Total Exclusions']]

df

## Causes of task failure-based exclusions

In [ ]:
query = f"SELECT project_id, step, stage, variant, info FROM task WHERE project_id = {project_id} AND status = 'FAILED'"
df = pd.read_sql_query(query, conn)

failure_types = [
    'Depth limit of 100 exceeded',
    'PC size limit exceeded',
    'Execution timeout exceeded',
    'No assertions found',
    'might be inherited',
    'contains nested types',
    'contains \\(static\\) initializers',
    'Failed to identify valid type for parameter',
    'has no @Test annotation',
    'AssertionFailedError',
    'java.lang.ArithmeticException: !!!div by 0',
    'java.lang.ClassNotFoundException: class not found: java.lang.NoSuchMethodException!!',
    'method arguments do not match with JPF\'s symbolic.method configuration',
    'ATHROW cannot be cast to gov.nasa.jpf.jvm.bytecode.JVMReturnInstruction',
    'INVOKESTATIC cannot be cast to gov.nasa.jpf.jvm.bytecode.JVMReturnInstruction',
    'java.lang.NullPointerException',
    'NEWARRAY: symbolic array length',
    'java.lang.ClassCastException',
    'Unable to transform operation',
    'java.io.FileNotFoundException',
    'Failed to collect input/output specification for unknown reason',
    'AssertionError',
    'no peer'
]

def categorize_failure(info):
    for failure_type in failure_types:
        if failure_type in info:
            return failure_type
    return '<Other>'

df['failure_type'] = df['info'].apply(categorize_failure)

failure_causes = df['failure_type'].value_counts().reset_index()
failure_causes = pd.concat([failure_causes, pd.DataFrame({'failure_type': ['<Total>'], 'count': [df['info'].count()]})])
failure_causes.sort_values(by='count', ascending=False).reset_index(drop=True)

#Other failures:
# mask = ~df['info'].str.contains('|'.join(failure_types))
# no_match_df = df[mask]
# no_match_df['info']

## Causes of filtering-based test exclusions

In [ ]:
query = f"SELECT project_id, exclusion_info FROM test WHERE project_id = {project_id} AND exclusion_info LIKE '%TestFilteringTask%'"
df = pd.read_sql_query(query, conn)

def process_info(info):
    lines = info.split('\n')
    result = {}
    for line in lines:
        # Split on first colon only
        split_idx = line.find(': ')
        if split_idx != -1:
            key = line[:split_idx]
            value = line[split_idx + 2:]  # +2 to skip ': '
            result[key] = 1 if value.startswith('REJECT') else 0
    return result

df['exclusion_info_dict'] = df['exclusion_info'].apply(process_info)

filter_df = pd.json_normalize(df['exclusion_info_dict'])

df = pd.concat([df, filter_df], axis=1)
df = df.drop(columns=['exclusion_info'])
df = df.drop(columns=['exclusion_info_dict'])

df.groupby('project_id').sum()

## Causes of filtering-based assertion exclusions

In [ ]:
query = f"SELECT project_id, exclusion_info FROM assertion WHERE project_id = {project_id} AND exclusion_info LIKE '%TestFilteringTask%'"
df = pd.read_sql_query(query, conn)

def process_info(info):
    lines = info.split('\n')
    result = {}
    for line in lines:
        # Split on first colon only
        split_idx = line.find(': ')
        if split_idx != -1:
            key = line[:split_idx]
            value = line[split_idx + 2:]  # +2 to skip ': '
            result[key] = 1 if value.startswith('REJECT') else 0
    return result

df['exclusion_info_dict'] = df['exclusion_info'].apply(process_info)

filter_df = pd.json_normalize(df['exclusion_info_dict'])

df = pd.concat([df, filter_df], axis=1)
df = df.drop(columns=['exclusion_info'])
df = df.drop(columns=['exclusion_info_dict'])

df.groupby('project_id').sum()

## Causes of filtering-based generalization exclusions

In [ ]:
query = f"SELECT project_id, exclusion_info FROM generalization WHERE project_id = {project_id} AND exclusion_info LIKE '%TestFilteringTask%'"
df = pd.read_sql_query(query, conn)

def process_info(info):
    lines = info.split('\n')
    result = {}
    for line in lines:
        # Split on first colon only
        split_idx = line.find(': ')
        if split_idx != -1:
            key = line[:split_idx]
            value = line[split_idx + 2:]  # +2 to skip ': '
            result[key] = 1 if value.startswith('REJECT') else 0
    return result

df['exclusion_info_dict'] = df['exclusion_info'].apply(process_info)

filter_df = pd.json_normalize(df['exclusion_info_dict'])

df = pd.concat([df, filter_df], axis=1)
df = df.drop(columns=['exclusion_info'])
df = df.drop(columns=['exclusion_info_dict'])

df.groupby('project_id').sum()

## Mutation testing results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, status, is_detected FROM pit_mutation_report WHERE project_id = {project_id}"
df = pd.read_sql_query(query, conn)
result = pd.DataFrame()

if not df.empty:
    df['variant'] = df['variant'].fillna('ORIGINAL')

    mutation_status_categories = ['SURVIVED', 'KILLED', 'TIMED_OUT', 'NO_COVERAGE', 'NON_VIABLE', 'MEMORY_ERROR', 'RUN_ERROR']
    mutation_status_categories = [c for c in mutation_status_categories if c in df['status'].unique()]
    df['status'] = pd.Categorical(df['status'], categories=mutation_status_categories)

    result = pd.pivot_table(
        df,
        index=['project_id', 'step', 'stage', 'variant'],
        columns='status',
        values='status',
        aggfunc='count',
        fill_value=0,
        observed=True
    )

    result['TOTAL'] = result[mutation_status_categories].sum(axis=1)
    result = result[['TOTAL'] + mutation_status_categories]

    result['DETECTED'] = pd.pivot_table(
        df,
        index=['project_id', 'step', 'stage', 'variant'],
        values='is_detected',
        aggfunc='sum',
        fill_value=0
    )

    # Calculate detected mutations safely
    killed = result['KILLED'] if 'KILLED' in result.columns else 0
    timed_out = result['TIMED_OUT'] if 'TIMED_OUT' in result.columns else 0
    no_coverage = result['NO_COVERAGE'] if 'NO_COVERAGE' in result.columns else 0

    # Calculate percentage detected of covered mutations
    covered_mutations = result['TOTAL'] - no_coverage
    detected_mutations = killed + timed_out
    result['% detected of covered'] = detected_mutations / covered_mutations.replace(0, 1)  # Avoid division by zero

    result.reset_index(inplace=True)

result

## Code coverage results pre-/post-generalization

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(instruction_missed), sum(instruction_covered), sum(branch_missed), sum(branch_covered) FROM jacoco_coverage_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements per generalization variant

In [ ]:
query = f"SELECT project_id, variant, count(DISTINCT step) AS steps, count(*) AS tasks, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, variant ORDER BY project_id"
df = pd.read_sql_query(query, conn)
df

## Runtime requirements per processing stage

In [ ]:
query = f"SELECT project_id, step, stage, variant, sum(runtime) AS runtime FROM task WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant ORDER BY project_id, step"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df

In [ ]:
df['step_stage'] = df['step'].astype(str) + "-" + df['stage'].astype(str) + "-" + df['variant'].astype(str)

df.plot(kind='bar', x='step_stage', y='runtime', legend=None, figsize=(12, 6))

plt.xlabel('Processing Stage')
plt.ylabel('Runtime (in seconds)')
plt.title('Runtime per Processing Stage')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Causes of test failures

In [ ]:
query = f"SELECT project_id, step, stage, variant, failure_type, count(*), sum(runtime) FROM junit_test_report WHERE project_id = {project_id} GROUP BY project_id, step, stage, variant, failure_type ORDER BY step, failure_type, variant"
df = pd.read_sql_query(query, conn)
df['variant'] = df['variant'].fillna('ORIGINAL')
df['failure_type'] = df['failure_type'].fillna('')
df